# Document Analysis & Visualization

Load a PDF or CSV, extract structured data via LLM, render interactive Plotly charts.

**Supported Providers:** OpenAI · Anthropic · Ollama — switch with a single variable.

## Overview

| Decision | Choice | Reason |
|---|---|---|
| API keys | `.env` file + `python-dotenv` | Standard local dev pattern, no Kaggle dependency |
| Provider switching | Single `LLM_PROVIDER` variable | One-line swap, no code changes elsewhere |
| Extraction method | `with_structured_output(Pydantic)` | Guaranteed JSON, type-validated |
| Visualization | Plotly Express | Interactive in Jupyter, minimal code |
| PDF loader | `PyMuPDFLoader` | Fastest community loader |

In [1]:
%pip install pymupdf plotly python-dotenv \
             langchain-openai langchain-anthropic langchain-ollama \
             langchain-community


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads from .env file if present

# ── Choose your provider ──────────────────────────────
LLM_PROVIDER = "ollama"   # "openai" | "anthropic" | "ollama"

# API keys — set here or in a .env file
OPENAI_API_KEY    = os.getenv("OPENAI_API_KEY", "sk-...")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "sk-ant-...")
# Ollama needs no API key — just run: `ollama pull llama3.2`

In [3]:
def build_llm(provider: str):
    if provider == "openai":
        from langchain_openai import ChatOpenAI
        os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
        return ChatOpenAI(model="gpt-4o-mini", temperature=0)

    elif provider == "anthropic":
        from langchain_anthropic import ChatAnthropic
        os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
        return ChatAnthropic(model="claude-sonnet-4-6", temperature=0)

    elif provider == "ollama":
        from langchain_ollama import ChatOllama
        return ChatOllama(model="llama3.2", temperature=0)

    else:
        raise ValueError(f"Unknown provider: {provider}. Use 'openai', 'anthropic', or 'ollama'.")

llm = build_llm(LLM_PROVIDER)
print(f"LLM ready: {LLM_PROVIDER}")

/Users/Dzmitry_Marudau/.local/share/mise/installs/python/3.14.0/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


/Users/Dzmitry_Marudau/.local/share/mise/installs/python/3.14.0/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LLM ready: ollama


In [4]:
from pathlib import Path
from typing import List, Optional

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pydantic import BaseModel, Field

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_core.prompts import ChatPromptTemplate

In [5]:
class DataPoint(BaseModel):
    label: str = Field(description="Name or label of the data point")
    value: float = Field(description="Numeric value")
    category: str = Field(description="Category or group this belongs to")
    unit: Optional[str] = Field(default=None, description="Unit of measurement if present")

class ExtractedReport(BaseModel):
    title: str = Field(description="Title or topic of the document")
    summary: str = Field(description="2-3 sentence summary of key findings")
    data_points: List[DataPoint] = Field(description="All numerical data found")
    key_metrics: dict = Field(description="Top-level KPIs as name:value pairs")

In [6]:
def load_document(file_path: str) -> str:
    path = Path(file_path)
    if path.suffix.lower() == ".pdf":
        docs = PyMuPDFLoader(file_path).load()
        return "\n\n".join(d.page_content for d in docs)
    elif path.suffix.lower() == ".csv":
        docs = CSVLoader(file_path).load()
        return "\n".join(d.page_content for d in docs)
    else:
        raise ValueError(f"Unsupported format: {path.suffix}. Use .pdf or .csv")

print("Document loader ready")

Document loader ready


In [7]:
structured_llm = llm.with_structured_output(ExtractedReport)

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a data extraction specialist.
Analyze the document and extract ALL numerical data points, metrics, and statistics.
Be precise with values. Capture units when present."""),
    ("human", "Document content:\n\n{content}\n\nExtract all structured data.")
])

extraction_chain = prompt | structured_llm
print("Extraction chain ready")

Extraction chain ready


In [8]:
FILE_PATH = "sample_report.csv"   # <- change to your file

content = load_document(FILE_PATH)
print(f"Loaded {len(content):,} characters from {FILE_PATH}")

result: ExtractedReport = extraction_chain.invoke({"content": content})

print(f"\nTitle:        {result.title}")
print(f"Summary:      {result.summary}")
print(f"Data points:  {len(result.data_points)}")
print(f"Key metrics:  {list(result.key_metrics.keys())}")

Loaded 1,532 characters from sample_report.csv



Title:        Financial Data
Summary:      Quarterly financial data for North America, Europe, and APAC regions.
Data points:  72
Key metrics:  ['North America', 'Europe', 'APAC']


In [9]:
df = pd.DataFrame([
    {"label": dp.label, "value": dp.value, "category": dp.category, "unit": dp.unit}
    for dp in result.data_points
])
print(df.to_string(index=False))

       label     value      category unit
      Region       3.0 North America None
 Revenue_USD 4200000.0       Q1 2024 None
Expenses_USD 2900000.0       Q1 2024 None
  Profit_USD 1300000.0       Q1 2024 None
  Growth_Pct      12.5       Q1 2024 None
   Headcount     320.0       Q1 2024 None
      Region       3.0 North America None
 Revenue_USD 4750000.0       Q2 2024 None
Expenses_USD 3100000.0       Q2 2024 None
  Profit_USD 1650000.0       Q2 2024 None
  Growth_Pct      13.1       Q2 2024 None
   Headcount     335.0       Q2 2024 None
      Region       3.0 North America None
 Revenue_USD 5100000.0       Q3 2024 None
Expenses_USD 3300000.0       Q3 2024 None
  Profit_USD 1800000.0       Q3 2024 None
  Growth_Pct       7.4       Q3 2024 None
   Headcount     350.0       Q3 2024 None
      Region       3.0 North America None
 Revenue_USD 5600000.0       Q4 2024 None
Expenses_USD 3500000.0       Q4 2024 None
  Profit_USD 2100000.0       Q4 2024 None
  Growth_Pct       9.8       Q4 20

In [10]:
fig = px.bar(
    df, x="label", y="value", color="category",
    title=f"{result.title} — Values by Label",
    text_auto=True, height=450
)
fig.update_layout(xaxis_tickangle=-40)
fig.show()

In [11]:
cat_totals = df.groupby("category")["value"].sum().reset_index()

fig = px.pie(
    cat_totals, values="value", names="category",
    title=f"{result.title} — Distribution by Category",
    hole=0.35
)
fig.show()

In [12]:
metrics_df = pd.DataFrame(
    list(result.key_metrics.items()), columns=["Metric", "Value"]
)

fig = go.Figure(data=[go.Table(
    header=dict(
        values=["Metric", "Value"],
        fill_color="#4C78A8",
        font=dict(color="white", size=13)
    ),
    cells=dict(values=[metrics_df["Metric"], metrics_df["Value"]])
)])
fig.update_layout(title="Key Metrics Summary")
fig.show()

In [13]:
print("=" * 60)
print(f"  {result.title}")
print("=" * 60)
print(f"\n{result.summary}")
print(f"\nCategories : {df['category'].unique().tolist()}")
print(f"Data points: {len(df)}")

  Financial Data

Quarterly financial data for North America, Europe, and APAC regions.

Categories : ['North America', 'Q1 2024', 'Q2 2024', 'Q3 2024', 'Q4 2024', 'Europe', 'APAC']
Data points: 72
